In [1]:
# Proje kökünü bul ve src/ klasörünü import yoluna ekle
import sys
from pathlib import Path

PROJE_KOKU = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "kontrol.py").exists())
sys.path.insert(0, str(PROJE_KOKU / "src"))

from asistan.ayarlar import ayar, model_getir

print("Yerel model:", ayar("OLLAMA_MODEL"), "@", ayar("OLLAMA_URL"))

Yerel model: qwen3.5:4b @ http://localhost:11434


In [2]:
SORU = "Kartımı kaybettim, ne yapmalıyım?"

## Yerel model (Ollama)

Model bilgisayarınızda çalışır, ücretsizdir ve soru bilgisayarınızdan dışarı çıkmaz.

In [6]:
import time

yerel_model = model_getir("yerel")

baslangic = time.perf_counter()
yerel_cevap = yerel_model.invoke(SORU)
yerel_sure = time.perf_counter() - baslangic

print(f"Süre: {yerel_sure:.1f} sn\n")
print(yerel_cevap.text)

Süre: 257.7 sn

Kartınızı kaybetmeniz durumunda paniklemek yerine **hızlı ve sırayla** aşağıdaki adımları izlemeniz gerekir. Bu süreç hem kartı geçici olarak bloklamayı hem de finansal kayıpları önlemeyi amaçlar:

### 1. En Önemli Adım: Hemen İletişime Geçin
Kartın bulunduğu yeri veya son görüldüğünüz yerdeki en yakın **banka şubesine** gidin veya bankanızın müşteri hizmetlerini arayın (genellikle 0850 numaralı bir hattadır). Kartınızı **bloke ettirin**. Bu işlem, kartı kimse tarafından kullanılmamasını sağlar ve dolandırıcılık riskini sıfıra indirir.

### 2. Kayıp Raporu Verin
Banka şubesine veya müşteri hizmetlerine giderek resmi bir **kayıp kart bildirimi (raporu)** verin. Bankanız, bu raporu alarak kartı geçici olarak dondurabilir ve gerekli görürse güvenlik önlemlerini artırır.

### 3. Yeni Kart İsteği Verin
Kartınız bloke edildiği için yeni bir kart numarası oluşturulması gerekir. Bankadan **yeni bir kart talep edin**. Genellikle:
*   Şubeden fiziksel olarak teslim alınabilir.
* 

##  Bulut model (Claude)

Model Anthropic'in sunucularında çalışır: daha güçlüdür, ama ücretlidir ve soru internet üzerinden dışarı gönderilir.
Bu hücre için `.env` dosyasında `ANTHROPIC_API_KEY` tanımlı olmalı.

In [7]:
import os

if os.getenv("ANTHROPIC_API_KEY"):
    bulut_model = model_getir("bulut")

    baslangic = time.perf_counter()
    bulut_cevap = bulut_model.invoke(SORU)
    bulut_sure = time.perf_counter() - baslangic

    print(f"Süre: {bulut_sure:.1f} sn\n")
    print(bulut_cevap.text)
else:
    bulut_cevap = None
    print("ANTHROPIC_API_KEY yok, bu hücre atlandı. Anahtarı .env dosyasına ekleyip tekrar deneyin.")

Süre: 8.6 sn

Çoğu zaman "kart" banka veya kredi kartı demektir, bu yüzden önce onu ele alıyorum.

## Banka veya kredi kartı kaybettiyseniz

1. **Kartı hemen bloke edin.** Şu yollardan biri yeterli:
   - Bankanızın mobil uygulaması veya internet bankacılığı (genelde "Kartı kapat/geçici blokaj" seçeneği vardır)
   - Bankanın 7/24 müşteri hizmetleri numarası (kartın arka yüzünde veya bankanın sitesinde yazar)
   - En yakın şube
2. **Son işlemlerinizi kontrol edin.** Size ait olmayan bir harcama görürseniz bankaya bildirip itiraz edin.
3. **Yeni kart isteyin.** Banka, eski kartı iptal edip yenisini gönderir veya şubeden teslim eder.
4. **Otomatik ödemeleri güncelleyin.** Abonelikler, faturalar ve online alışveriş sitelerindeki kayıtlı kart bilgilerini yeni kartla değiştirin.
5. **Şifrenizi değiştirmeyi düşünün.** Özellikle kartla birlikte not veya kimlik kaybettiyseniz bu önemli.

Karta kayıtlı temassız ödeme veya mobil cüzdan varsa onları da kontrol edin.

## Başka bir kart olabilir mi?


In [8]:
sonuclar = [("Yerel", yerel_cevap, yerel_sure)]
if bulut_cevap is not None:
    sonuclar.append(("Claude", bulut_cevap, bulut_sure))

print(f"{'Model':<8} {'Süre (sn)':>10} {'Girdi token':>12} {'Çıktı token':>12} {'Karakter':>9}")
for ad, cevap, sure in sonuclar:
    token = cevap.usage_metadata
    print(f"{ad:<8} {sure:>10.1f} {token['input_tokens']:>12} {token['output_tokens']:>12} {len(cevap.text):>9}")

Model     Süre (sn)  Girdi token  Çıktı token  Karakter
Yerel         257.7           25          626      2088
Claude          8.6           29          868      1497


## Tartışma

Aşağıdaki soruları düşünün. Cevaplarınızı bu hücrenin altına yeni bir markdown hücresi ekleyerek yazın (Görev 1 teslimi).

1. **Hız:** Hangi model daha hızlı cevap verdi? Hücreleri birkaç kez çalıştırın: süreler değişiyor mu? Değişiyorsa neden?
2. **Kalite:** Hangi cevabı olduğu gibi bir banka müşterisine gönderirdiniz? Cevaplarda yanlış ya da uydurma bir bilgi var mı?
3. **Bağlam:** İki model de Örnek Bank'ın kayıp kart prosedürünü biliyor mu? Bilmiyorsa bu bilgiyi modele nasıl verebiliriz?
4. **Maliyet ve gizlilik:** Soruda müşterinin adı ve kart numarası olsaydı, hangi modele göndermek risk olurdu? Günde 10.000 talep gelen bir bankada hangi modeli, neden seçerdiniz?

## Görev 1 - Cevaplar

1. Bulut model, yerel modele göre daha hızlı cevap üretmiştir. Tablolara bakarak bulut modelin yanıt süresi daha stabil gözükmektedir. . Her iki modelin de kaynak veya sunucu yoğunluğuna bağlı olarak cevap süresi değişebilir. 

    **Çalıştırma 1 - Sonuçlar**
    | Model | Süre (sn) | Girdi Token | Çıktı Token | Karakter |
    |---|---:|---:|---:|---:|
    | Yerel | 225.7 | 25 | 626 | 2088 |
    | Claude | 8.5 | 29 | 901 | 1522 |
    
    **Çalıştırma 2 - Sonuçlar**
    | Model | Süre (sn) | Girdi Token | Çıktı Token | Karakter |
    |---|---:|---:|---:|---:|
    | Yerel | 257.7 | 25 | 626 | 2088 |
    | Claude | 8.6 | 29 | 868 | 1497 |

2. Her iki cevabı da kaynak doğrulaması yapılmadan banka müşterisine doğrudan gönderilmesi uygun değildir. Bulut modeli, kullanıcının bahsettiği kartın kredi kartı olduğunu varsayarak cevap vermiş, ancak yemek kartı ve kimlik kartı gibi farklı kart türlerinden de bahsetmiştir. Yerel modelin cevabı daha doğru görünsede herhangi bir kaynak göstermeden kesin bilgiler içermektedir. Bu nedenle her iki cevap da doğrulanmalıdır.

3. İki modelin de Örnek Bank'ın kayıp prosedürü hakkında bilgisi bulunmamaktadır. RAG(Retrieval Augmented Generation) mimarisiyle modellerin ilgili bankanın kayıp prosedürü dokümanlarından soru-cevap yapması sağlanabilir.

4. Sorularda müşterilere ait hassas veri bulunuyorsa bu veriyi bulut modele göndermek güvenlik ve gizlilik açısından risk oluşturabilir. Çünkü gönderilen bilgiler kurum dışı bi hizmet sağlayıcının alt yapısında kullanılabilir. 
Günde 10.000 talep gelen bir bankada yerel modeli tercih ederdim. Bulut modeli daha hızlı cevap üretebilme avantajı olsada güvenlik ve gizliliğin sağlanması kurumlar için daha hassas bir konudur. Ek olarak bulut model kullanımı bağımlılığı ve maliyeti de arttırır. 
Yerel model için donanım kaynaklarını arttırılması ve model optimizasyonları yapılması ile cevap kalitesi ve performansı iyileştirilebilir. 